# 📊 Auditoría del Mercado Laboral Tecnológico
### Proyecto End-to-End de Ciencia de Datos

---

| Atributo | Detalle |
|---|---|
| **Dataset** | `DataAnalyst.csv` (Fuente: Kaggle / Glassdoor) |
| **Volumen Original** | 2,253 registros × 16 columnas |
| **Motor Analítico** | Python 3 · Pandas · NumPy · SQLite3 |
| **Alcance** | Ingesta → Limpieza → Persistencia → Análisis Exploratorio |

**Objetivo General:** Ejecutar un pipeline completo de análisis de datos sobre un conjunto de ofertas laborales del sector tecnológico, desde la ingesta cruda hasta la extracción de insights accionables mediante consultas SQL.

---

### Estructura del Pipeline

```
Fase 1 ─── Ingesta y Diagnóstico ──────────▶ Carga del CSV + Radiografía estructural
  │
Fase 2 ─── Limpieza y Normalización ────────▶ Regex, Type Casting, Imputación, Purga
  │
Fase 3 ─── Persistencia Relacional ─────────▶ Migración a SQLite (base de datos física)
  │
Fase 4 ─── Análisis Exploratorio (EDA) ─────▶ Consultas SQL: tendencia central, segmentación
```

---
# 📗 Fase 1: Ingesta y Diagnóstico

**Objetivo:** Cargar el conjunto de datos en memoria como un objeto `DataFrame` de Pandas, y ejecutar una auditoría estructural para identificar anomalías antes de cualquier transformación.

---

## 1.1 Importación de Librerías e Ingesta del Dataset

Instanciamos el motor de Pandas (`pd`) y NumPy (`np`) para el manejo de estructuras tabulares y operaciones numéricas. Luego cargamos el archivo CSV en un `DataFrame`, que es la estructura de datos central de Pandas — una tabla bidimensional en memoria volátil.

In [ ]:
import pandas as pd
import numpy as np

# Ingesta: lectura del archivo de texto plano (CSV) hacia memoria
df = pd.read_csv('./data/DataAnalyst.csv')

## 1.2 Diagnóstico de Metadatos (`df.info()`)

Ejecutamos `.info()` para obtener la **radiografía estructural** del DataFrame. Este método nos revela:
- Número total de filas y columnas
- Nombre y tipo de dato (`Dtype`) de cada columna
- Cantidad de valores no nulos por columna
- Consumo de memoria

> **¿Por qué es importante?** Porque nos permite detectar problemas *antes* de ver los datos: columnas con tipos incorrectos (ej. salarios como texto), índices residuales, y falsos positivos de integridad.

In [ ]:
df.info()

### 🔍 Hallazgos del Diagnóstico

| Hallazgo | Descripción | Impacto |
|---|---|---|
| **Índice residual** | Columna `Unnamed: 0` — artefacto del proceso de scraping | Ocupa memoria sin aportar información |
| **Tipos incorrectos** | `Salary Estimate` es `object` (texto), no numérico | Imposibilita cálculos matemáticos sobre salarios |
| **Nulos ocultos** | Valores como `-1`, `"Unknown"` no son detectados como `NaN` | Falsos positivos de integridad — el DataFrame "parece" completo |

## 1.3 Alteración Estructural — Eliminación del Índice Residual

La columna `Unnamed: 0` es un artefacto del scraping original (un índice del CSV que se importó como columna). La eliminamos sobre el **eje de columnas** (`axis=1`).

> **Concepto clave — Ejes en Pandas:**
> - `axis=0` → Opera sobre **filas** (vertical)
> - `axis=1` → Opera sobre **columnas** (horizontal)

In [ ]:
# Eliminación de la columna artefacto sobre el eje de columnas
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

## 1.4 Análisis Visual de Anomalías — Subsetting

Extraemos un subconjunto bidimensional (`df[['col1', 'col2', 'col3']]`) para aislar las variables que presentan anomalías visibles. Esta técnica se llama **Subsetting Estructural** — usamos una lista de Python dentro de los corchetes del DataFrame.

> **¿Qué buscamos?** Evidencia empírica de la "suciedad" textual en columnas que *deberían* ser numéricas.

In [ ]:
# Subsetting: aislamos las variables sospechosas
df[['Salary Estimate', 'Founded', 'Rating']].head()

### 📋 Evidencia Empírica

La columna `Salary Estimate` contiene:
- **Caracteres parásitos:** `$`, `(Glassdoor est.)`
- **Multiplicadores implícitos:** `K` (miles)
- **Delimitadores de rango:** `-` (mínimo-máximo)
- **Ejemplo:** `$37K-$66K (Glassdoor est.)` → Necesitamos extraer `37000` y `66000`

Adicionalmente, `Company Name` contiene saltos de línea (`\n`) anidados con el rating.

---
### ✅ Fase 1 Completada
Se identificaron todas las anomalías estructurales. El DataFrame está listo para la fase de limpieza.

---
# 📙 Fase 2: Limpieza y Normalización (Data Wrangling)

**Objetivo:** Saneamiento estructural completo — transformación vectorial de cadenas de texto, corrección de tipos de datos, estandarización de anomalías y resolución integral del déficit de información (nulos).

---

## 2.1 Extracción Estructural con Expresiones Regulares (Regex)

Deconstruimos la variable `Salary Estimate` para aislar los límites numéricos del rango salarial.

**Técnica:** Implementación de un autómata de estado finito mediante Regex, aplicado de forma **vectorial** a través del accesor `.str` de Pandas.

```
Patrón:  \$(\d+)K-\$(\d+)K
         │  │      │  │
         │  │      │  └── Captura dígitos del máximo
         │  │      └───── Símbolo $ literal (escapado)
         │  └──────────── Captura dígitos del mínimo  
         └─────────────── Símbolo $ literal (escapado)
```

In [ ]:
# Extracción de los límites salariales mediante Regex
df[['Min_Salary', 'Max_Salary']] = df['Salary Estimate'].str.extract(r'\$(\d+)K-\$(\d+)K')

# Validación visual: comparamos la columna original con las nuevas
df[['Salary Estimate', 'Min_Salary', 'Max_Salary']].head()

## 2.2 Conversión de Tipos (Type Casting) y Escalado Numérico

Los valores extraídos por Regex son `object` (texto). Los mutamos hacia `float64` — un tipo numérico tolerante a valores nulos (`NaN`) — y restituimos la escala multiplicando por 1,000 (el `K` que removimos).

In [ ]:
# Type Casting: str → float64, y escalado ×1000
df['Min_Salary'] = df['Min_Salary'].astype(float) * 1000
df['Max_Salary'] = df['Max_Salary'].astype(float) * 1000

# Verificación de tipos resultantes
df[['Min_Salary', 'Max_Salary']].dtypes

## 2.3 Ingeniería de Características (Feature Engineering)

Derivamos una nueva variable: el **promedio salarial** (`Avg_Salary`), que será nuestra variable objetivo para el análisis exploratorio. Luego eliminamos la columna original `Salary Estimate` — ya es redundante.

> **Feature Engineering** = Crear nuevas variables a partir de las existentes para enriquecer el poder analítico del dataset.

In [ ]:
# Derivación del salario promedio
df['Avg_Salary'] = (df['Max_Salary'] + df['Min_Salary']) / 2

# Purga de la columna original (redundancia estructural)
df.drop('Salary Estimate', inplace=True, axis=1)

## 2.4 Saneamiento Textual — `Company Name`

La columna `Company Name` tiene artefactos de scraping: saltos de línea (`\n`) seguidos del rating de la empresa. Usamos el accesor `.str.split()` para aislar solo el nombre.

In [ ]:
# Eliminación de artefactos de scraping (\n + rating) en Company Name
df['Company Name'] = df['Company Name'].str.split('\n', expand=True)[0]

## 2.5 Estandarización de Valores Centinela

El dataset del scraping usa valores "centinela" para indicar datos ausentes: `-1` (numérico), `'-1'` (texto) y `'Unknown'`. Estos no son reconocidos por Pandas como nulos nativos. Los transmutamos a `NaN` para que el motor de Pandas los trate correctamente.

> **Valores centinela** = Valores arbitrarios inyectados por el proceso de extracción para representar "sin datos". Son peligrosos porque pasan desapercibidos en análisis estadísticos.

In [ ]:
# Transmutación de valores centinela a NaN nativo
df.replace([-1, '-1', 'Unknown'], np.nan, inplace=True)

## 2.6 Diagnóstico de Nulidad Post-Estandarización

Ahora que los valores centinela son `NaN`, ejecutamos `.isna().sum()` para medir el déficit real de información por columna.

In [ ]:
# Mapa de nulidad real del DataFrame
df.isna().sum()

### 📊 Análisis del Déficit de Información

| Categoría | Variables | Estrategia |
|---|---|---|
| **Escasez crítica (irrecuperable)** | `Competitors` (1732), `Easy Apply` (2173) | Eliminación total de la variable |
| **Escasez moderada (categóricas)** | `Headquarters`, `Size`, `Founded`, `Type of ownership`, `Industry`, `Sector`, `Revenue` | Imputación por constante: `"No Especificado"` |
| **Escasez moderada (numérica)** | `Rating` (272) | Imputación estadística: mediana |
| **Escasez mínima (cuantitativa)** | `Min/Max/Avg_Salary` (1-2 registros) | Eliminación del registro (Listwise Deletion) |

## 2.7 Reducción de Dimensionalidad — Variables Irrecuperables

Las variables `Easy Apply` y `Competitors` tienen más del 76% de nulos. Imputarlas generaría **sesgo sintético** — datos inventados que distorsionan el análisis. Las purgamos.

In [ ]:
# Eliminación de variables con escasez crítica (>76% nulos)
df.drop(columns=['Easy Apply', 'Competitors'], inplace=True)

# Verificación del espacio de columnas resultante
df.columns

## 2.8 Imputación por Constante Categórica

Para las variables nominales/temporales con escasez moderada, asignamos la constante `"No Especificado"`. Esto preserva la varianza empírica (no inventamos categorías) y mantiene los registros completos para análisis posteriores.

In [ ]:
# Imputación de variables categóricas con constante explícita
cols_cat = ['Headquarters', 'Size', 'Founded', 'Type of ownership',
            'Industry', 'Sector', 'Revenue']
df[cols_cat] = df[cols_cat].fillna('No Especificado')

# Verificación parcial
df.isna().sum()

## 2.9 Imputación Estadística Univariante — `Rating`

Para la variable continua `Rating`, calculamos el **centroide gravitacional resistente a outliers**: la **mediana**. Usamos la mediana en lugar de la media porque es robusta frente a valores atípicos.

> **Media vs Mediana:** La media se distorsiona con un solo valor extremo. La mediana representa el valor central de la distribución ordenada.

In [ ]:
# Imputación de Rating con la mediana (robusta a outliers)
mediana_rating = df['Rating'].median()
print(f'Mediana calculada: {mediana_rating}')

df['Rating'] = df['Rating'].fillna(mediana_rating)

## 2.10 Erradicación de Escasez Mínima (Listwise Deletion)

Quedan 1-2 registros con ausencia de datos salariales (`Min_Salary`, `Max_Salary`, `Avg_Salary`). Como es una proporción insignificante del dataset (<0.1%), los eliminamos directamente. Luego reconstruimos el índice secuencial para evitar fragmentación.

In [ ]:
# Inspección del registro con salarios faltantes
df[df['Avg_Salary'].isna()]

In [ ]:
# Purga matricial definitiva + reconstrucción del índice
df = df.dropna()
df = df.reset_index(drop=True)

# Validación final: 0 nulos en todas las columnas
print(f'Dimensiones finales: {df.shape}')
print(f'\nMapa de nulidad final:')
df.isna().sum()

---
### ✅ Fase 2 Completada

La matriz de características ha alcanzado **pureza estructural nula** (0 valores ausentes) y **contigüidad referencial** (índice secuencial sin fragmentación).

| Métrica | Antes | Después |
|---|---|---|
| Columnas | 16 | 15 |
| Registros | 2,253 | 2,251 |
| Nulos totales | ~5,700 | 0 |
| Salarios | Texto (`$37K-$66K...`) | Numérico (`float64`) |

---
# 📕 Fase 3: Persistencia Relacional (SQLite)

**Objetivo:** Migrar la matriz de características desde la memoria volátil (Pandas DataFrame) hacia un motor de base de datos relacional (SQLite) para garantizar persistencia física, aplicar rigor transaccional y habilitar consultas analíticas con SQL.

> **¿Por qué SQLite?** Es un motor relacional embebido (no requiere servidor), nativo de Python, ideal para proyectos analíticos. Los datos se persisten en un archivo `.db` en el sistema de archivos local.

---

## 3.1 Establecimiento del Túnel Transaccional

Importamos el controlador nativo `sqlite3` e instanciamos una conexión. Si el archivo de base de datos no existe, SQLite lo crea automáticamente.

In [ ]:
import sqlite3

# Establecimiento de la conexión (crea el archivo físico si no existe)
conexion = sqlite3.connect('./data/DataAnalyst.db')

## 3.2 Migración Matricial — Exportación a SQL

Usamos `.to_sql()` de Pandas para transferir los registros del DataFrame a una tabla relacional. Los parámetros clave:
- `'ofertas_laborales'` → Nombre de la tabla destino
- `if_exists='replace'` → Sobrescribe si ya existe (idempotencia)
- `index=False` → No exporta el índice de Pandas como columna

In [ ]:
# Migración del DataFrame a tabla SQL
df.to_sql('ofertas_laborales', con=conexion, if_exists='replace', index=False)

## 3.3 Auditoría de Integridad

Validamos que la migración fue completa ejecutando un `SELECT COUNT(*)` a través de un cursor SQL. Debemos obtener **2,251 registros** — exactamente los mismos que en el DataFrame limpio.

In [ ]:
# Auditoría: conteo de registros en la tabla SQL
cursor = conexion.cursor()
resultado = cursor.execute('SELECT COUNT(*) FROM ofertas_laborales').fetchone()
print(f'Registros persistidos en SQLite: {resultado[0]}')

## 3.4 Cierre del Túnel Transaccional

Cerramos la conexión para consolidar el archivo físico y liberar el descriptor del sistema operativo. Sin este paso, el archivo `.db` podría quedar bloqueado.

In [ ]:
# Cierre y consolidación del archivo físico
conexion.close()

---
### ✅ Fase 3 Completada

Los datos limpios se encuentran persistidos en `./data/DataAnalyst.db`, tabla `ofertas_laborales`. La base de datos está lista para consultas analíticas SQL.

```
proyecto_auditoria_empleo/
├── data/
│   ├── DataAnalyst.csv      ← Datos crudos originales
│   └── DataAnalyst.db       ← Datos limpios persistidos (SQLite)
└── fase1_ingesta.ipynb      ← Este notebook
```

---
# 📘 Fase 4: Análisis Exploratorio de Datos (EDA Relacional)

**Objetivo:** Ejecutar análisis exploratorio cuantitativo directamente sobre el motor SQLite mediante álgebra relacional (SQL), evaluando medidas de tendencia central, dispersión, distribución geográfica y concentración de demanda laboral.

> **¿Por qué SQL y no Pandas?** Ambos son válidos, pero SQL nos entrena en pensamiento relacional — un skill fundamental en ciencia de datos. Además, las queries SQL son más legibles para stakeholders no técnicos.

---

## 4.1 Reconexión y Validación Estructural

Reabrimos la conexión a la base de datos persistida y validamos que los datos son accesibles.

In [ ]:
import sqlite3
import pandas as pd

# Reconexión al motor SQLite
conexion = sqlite3.connect('./data/DataAnalyst.db')

# Validación: primeros 5 registros de la tabla
query_validacion = 'SELECT * FROM ofertas_laborales LIMIT 5'
df_validacion = pd.read_sql_query(query_validacion, conexion)
df_validacion

## 4.2 Medidas de Tendencia Central Globales

Calculamos los estadísticos descriptivos fundamentales sobre la variable objetivo (`Avg_Salary`):
- **Media (AVG):** Promedio aritmético global
- **Mínimo (MIN):** Límite inferior del rango salarial
- **Máximo (MAX):** Límite superior del rango salarial
- **Rango:** MAX - MIN → Mide la dispersión total

In [ ]:
query_salarios = """
SELECT 
    ROUND(AVG(Avg_Salary), 2)  AS Promedio_Global,
    ROUND(MIN(Avg_Salary), 2)  AS Salario_Minimo_Global,
    ROUND(MAX(Avg_Salary), 2)  AS Salario_Maximo_Global,
    ROUND(MAX(Avg_Salary) - MIN(Avg_Salary), 2) AS Rango_Salarial
FROM ofertas_laborales
"""
df_salarios = pd.read_sql_query(query_salarios, conexion)
df_salarios

### 🔎 Interpretación

- **Media salarial global:** ~$72,121 USD — Esta es la compensación promedio para roles de Data Analyst en el mercado tech estadounidense.
- **Rango:** ~$116,500 USD — Indica **alta dispersión** en las bandas de compensación, lo que sugiere que factores como ubicación, sector y nivel de experiencia impactan significativamente.

## 4.3 Segmentación Bivariante por Sector

Particionamos los datos mediante `GROUP BY Sector` y aplicamos un **filtro de significancia muestral** con `HAVING COUNT(*) >= 30`. Esto elimina sectores con muestras muy pequeñas que podrían generar promedios engañosos (varianza espuria).

> **¿Por qué n ≥ 30?** Por el Teorema del Límite Central: con muestras de 30+, la distribución de la media muestral se aproxima a una normal, lo que hace confiables las comparaciones.

In [ ]:
query_sector = """
SELECT 
    Sector,
    ROUND(AVG(Avg_Salary), 2) AS Promedio_Sector,
    COUNT(*) AS Volumen_Ofertas
FROM ofertas_laborales
GROUP BY Sector
HAVING COUNT(*) >= 30
ORDER BY Promedio_Sector DESC
LIMIT 10
"""
df_sector = pd.read_sql_query(query_sector, conexion)
df_sector

### 🔎 Hallazgos Sectoriales

| Sector | Promedio USD | Volumen | Interpretación |
|---|---|---|---|
| **Biotech & Pharmaceuticals** | ~$83,106 | 33 | Prima por especialización científica |
| **Accounting & Legal** | ~$75,221 | 43 | Demanda regulatoria de análisis de datos |
| **Information Technology** | ~$74,247 | **570** | Mayor volumen de mercado |
| **Health Care** | ~$72,808 | 151 | Crecimiento post-pandemia |
| **Business Services** | ~$72,273 | **523** | Segundo mayor volumen |

## 4.4 Segmentación Espacial por Ubicación (Location)

Evaluamos la distribución geográfica con el mismo umbral muestral (n ≥ 30) para identificar polos de alta compensación y alta absorción laboral.

In [ ]:
query_ubicacion = """
SELECT 
    Location,
    ROUND(AVG(Avg_Salary), 2) AS Promedio_Ubicacion,
    COUNT(*) AS Volumen_Ofertas
FROM ofertas_laborales
GROUP BY Location
HAVING COUNT(*) >= 30
ORDER BY Promedio_Ubicacion DESC
LIMIT 10
"""
df_ubicacion = pd.read_sql_query(query_ubicacion, conexion)
df_ubicacion

### 🔎 Hallazgos Geográficos

**Polos de Alta Compensación** (prima por aglomeración y costo de vida):
1. **San Jose, CA** → ~$108,855 USD (n=38) — Epicentro de Silicon Valley
2. **San Francisco, CA** → ~$99,273 USD (n=119) — Hub tecnológico premium
3. **San Diego, CA** → ~$91,460 USD (n=62) — Ecosistema biotech

**Polo de Alta Absorción Laboral:**
- **New York, NY** → ~$71,235 USD (n=**310**) — Mayor volumen de demanda del dataset, pero con salario promedio moderado

## 4.5 Demanda y Concentración por Perfil (Job Title)

Identificamos los títulos laborales más frecuentes y sus salarios asociados. Esto revela la **moda dominante** del mercado y nos permite detectar problemas de normalización textual.

In [ ]:
query_titulos = """
SELECT 
    "Job Title",
    COUNT(*) AS Volumen_Ofertas,
    ROUND(AVG(Avg_Salary), 2) AS Promedio_Salario
FROM ofertas_laborales
GROUP BY "Job Title"
ORDER BY Volumen_Ofertas DESC
LIMIT 10
"""
df_titulos = pd.read_sql_query(query_titulos, conexion)
df_titulos

### 🔎 Diagnóstico de Calidad en Títulos

- **Moda dominante:** `"Data Analyst"` con 404 ofertas (~$72,577 USD promedio)
- **Fragmentación léxica detectada:** Se observan variantes del mismo perfil que deberían unificarse:
  - `"Senior Data Analyst"` vs `"Sr. Data Analyst"` → Mismo nivel, diferente nomenclatura
  - `"Junior Data Analyst"` vs `"Data Analyst Junior"` → Orden invertido

> 📝 **Nota:** La normalización de estas variantes léxicas sería un paso natural de refinamiento si el análisis se extendiera hacia modelado predictivo o dashboards ejecutivos.

## 4.6 Cierre y Liberación de Recursos

Destruimos el túnel transaccional para evitar bloqueos lógicos sobre el archivo físico.

In [ ]:
# Cierre definitivo de la conexión SQLite
conexion.close()
print('Conexión cerrada exitosamente.')

---
## Acta de Cierre

```
╔══════════════════════════════════════════════════════════════════════╗
║                                                                      ║
║   PROYECTO: Auditoría del Mercado Laboral Tecnológico                ║
║   ESTADO:   ✅ FINALIZADO                                            ║
║                                                                      ║
║   Fases Completadas:                                                 ║
║     ✅ Fase 1 — Ingesta y Diagnóstico                                ║
║     ✅ Fase 2 — Limpieza y Normalización (Data Wrangling)            ║
║     ✅ Fase 3 — Persistencia Relacional (SQLite)                     ║
║     ✅ Fase 4 — Análisis Exploratorio de Datos (EDA Relacional)      ║
║                                                                      ║
║   Entregables Verificados:                                           ║
║     ✅ DataFrame limpio (0 nulos, tipos correctos)                   ║
║     ✅ Base de datos SQLite persistida (2,251 registros)             ║
║     ✅ KPIs extraídos mediante consultas SQL                         ║
║     ✅ Documentación completa por fase                               ║
║     ✅ Reporte descriptivo final                                     ║
║                                                                      ║
║   Fecha de Cierre: 17 de Agosto, 2026                                ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

> *El ciclo de cuatro fases ha sido completado satisfactoriamente. Todos los objetivos definidos en la planificación original fueron alcanzados.*